# Analysing a Randomised Experiment: Class Size and Learning (Tennessee STAR)

**Question:** Did placing kindergarten students in small classes improve their maths scores, and by how much?

**Data:** The Tennessee STAR experiment (1985–1989), in which students were randomly assigned to small classes, regular classes, or regular classes with a teaching aide. Because assignment was random, differences in outcomes between groups can be interpreted as the effect of class size.

## Step 1: Loading and exploring the data

Before any analysis, I check the size of the dataset, what variables it contains, and how many students are in each group.

In [1]:
import pandas as pd                 # library for working with tables of data
import statsmodels.api as sm        # statistics library; also lets us download public datasets

# Download the STAR dataset (one row per student)
star = sm.datasets.get_rdataset("STAR", "AER").data

# Show the number of rows (students) and columns (variables)
print("Rows, columns:", star.shape)

# Show the first five rows to see what the data looks like
star.head()

Rows, columns: (11598, 47)


,gender,ethnicity,birth,stark,star1,star2,star3,readk,read1,read2,...,tethnicity2,tethnicity3,systemk,system1,system2,system3,schoolidk,schoolid1,schoolid2,schoolid3
rownames,,,,,,,,,,,,,,,,,,,,,
1122,female,afam,1979 Q3,NaN,NaN,NaN,regular,NaN,NaN,NaN,...,NaN,cauc,NaN,NaN,NaN,22.0,NaN,NaN,NaN,54.0
1137,female,cauc,1980 Q1,small,small,small,small,447.0,507.0,568.0,...,cauc,cauc,30.0,30.0,30.0,30.0,63.0,63.0,63.0,63.0
1143,female,afam,1979 Q4,small,small,regular+aide,regular+aide,450.0,579.0,588.0,...,afam,cauc,11.0,11.0,11.0,11.0,20.0,20.0,20.0,20.0
1160,male,cauc,1979 Q4,NaN,NaN,NaN,small,NaN,NaN,NaN,...,NaN,cauc,NaN,NaN,NaN,6.0,NaN,NaN,NaN,8.0
1183,male,afam,1980 Q1,regular+aide,NaN,NaN,NaN,439.0,NaN,NaN,...,NaN,NaN,11.0,NaN,NaN,NaN,19.0,NaN,NaN,NaN


In [2]:
# List all the variable names
print(star.columns.tolist())

# Count how many students were assigned to each class type in kindergarten
# (dropna=False also counts students who weren't in the study in kindergarten)
print(star["stark"].value_counts(dropna=False))

['gender', 'ethnicity', 'birth', 'stark', 'star1', 'star2', 'star3', 'readk', 'read1', 'read2', 'read3', 'mathk', 'math1', 'math2', 'math3', 'lunchk', 'lunch1', 'lunch2', 'lunch3', 'schoolk', 'school1', 'school2', 'school3', 'degreek', 'degree1', 'degree2', 'degree3', 'ladderk', 'ladder1', 'ladder2', 'ladder3', 'experiencek', 'experience1', 'experience2', 'experience3', 'tethnicityk', 'tethnicity1', 'tethnicity2', 'tethnicity3', 'systemk', 'system1', 'system2', 'system3', 'schoolidk', 'schoolid1', 'schoolid2', 'schoolid3']
stark
NaN             5273
regular+aide    2231
regular         2194
small           1900
Name: count, dtype: int64


**What the data shows:** The dataset contains 11,598 students, of whom 6,325 were in the experiment in kindergarten: 1,900 in small classes, 2,194 in regular classes and 2,231 in regular classes with an aide. The remaining 5,273 students joined in later grades and are excluded, since this analysis focuses on kindergarten outcomes.

## Step 2: Balance check

Before estimating the effect, I check whether the three groups had similar student characteristics. If randomisation worked, the groups should be similar, so any later difference in scores can be attributed to class size.

In [3]:
# Keep only students who were in the experiment in kindergarten
k = star[star["stark"].notna()].copy()

# Turn characteristics into 1/0 variables so we can calculate the share in each group
# (students with missing information stay missing rather than counted as 0)
k["female"] = k["gender"].map({"female": 1, "male": 0})
k["free_lunch"] = k["lunchk"].map({"free": 1, "non-free": 0})
k["african_american"] = (k["ethnicity"] == "afam").where(k["ethnicity"].notna()).astype(float)

# Share of each characteristic in each class type
balance = k.groupby("stark")[["female", "free_lunch", "african_american"]].mean().round(3)
print(balance)

              female  free_lunch  african_american
stark                                             
regular        0.490       0.477             0.324
regular+aide   0.483       0.503             0.338
small          0.486       0.471             0.312


**Balance result:** The three groups are very similar before the experiment. The share of girls (48–49%), students receiving free lunch (47–50%) and African American students (31–34%) differs by no more than about three percentage points between groups. This suggests randomisation worked, so differences in later scores can reasonably be attributed to class size.

## Step 3: Effect of small classes on maths scores

I compare small and regular classes, measuring scores in standard deviations of the regular-class group. I estimate the effect two ways: a simple comparison, and a comparison within schools (since randomisation happened within schools). Standard errors are clustered by school, because students in the same school are not independent.


In [4]:
import statsmodels.formula.api as smf   # lets us write regressions as simple formulas

# Keep small and regular classes, with a maths score and a school ID
a = k[k["stark"].isin(["small", "regular"])].dropna(subset=["mathk", "schoolidk"]).copy()

# 1 = small class, 0 = regular class
a["small"] = (a["stark"] == "small").astype(int)

# Convert maths scores into standard deviations, using the regular-class group as the reference
regular_scores = a.loc[a["small"] == 0, "mathk"]
a["math_sd"] = (a["mathk"] - regular_scores.mean()) / regular_scores.std()

# Model 1: simple comparison of small vs regular classes
m1 = smf.ols("math_sd ~ small", data=a).fit(
    cov_type="cluster", cov_kwds={"groups": a["schoolidk"]})

# Model 2: comparison within the same school (school fixed effects)
m2 = smf.ols("math_sd ~ small + C(schoolidk)", data=a).fit(
    cov_type="cluster", cov_kwds={"groups": a["schoolidk"]})

# Print the effect, 95% confidence interval and p-value for each model
print("Students in analysis:", len(a))
for name, m in [("Simple comparison", m1), ("Within-school comparison", m2)]:
    effect = m.params["small"]
    low, high = m.conf_int().loc["small"]
    p = m.pvalues["small"]
    print(f"{name}: {effect:.3f} SD (95% CI {low:.3f} to {high:.3f}), p = {p:.4f}")


Students in analysis: 3794
Simple comparison: 0.162 SD (95% CI 0.053 to 0.271), p = 0.0035
Within-school comparison: 0.185 SD (95% CI 0.070 to 0.301), p = 0.0017


**Result:** Students in small classes scored **0.16 SD higher** in maths than students in regular classes (95% CI: 0.05 to 0.27, p = 0.004). When comparing students within the same school, the effect is **0.19 SD** (95% CI: 0.07 to 0.30, p = 0.002). Both estimates are statistically significant.

The within-school estimate is the preferred result, because randomisation took place within schools and this comparison removes differences between schools.

**Limitations:**
- About 300 of the 4,094 students in small or regular classes (around 7%) have no kindergarten maths score. If these students differ systematically between groups, the estimate could be slightly biased.
- This analysis covers kindergarten maths only, and the study took place in the United States in the 1980s, so results may not transfer directly to other contexts.
- The "regular class with aide" group was not analysed here and could be compared in a follow-up.

## Summary for non-technical readers

**What was tested:** Whether putting kindergarten students in smaller classes (13–17 students instead of 22–25) improves their maths results.

**What we found:** Yes. Students in small classes scored noticeably higher, an improvement of about 0.19 standard deviations. In education research, this is a meaningful effect for a single change to how classes are organised.

**How confident are we:** Fairly confident. Students were assigned to classes at random, and the groups were very similar before the experiment started, so the difference is very likely caused by class size rather than by who was in each class.

**What I would look at next:**
1. Whether the effect is larger for some students, for example those from low-income families.
2. Whether the improvement lasts into later grades.
3. Whether adding a teaching aide to a regular class has a similar effect at lower cost.